# 📈 Notebook 2: Exploratory Data Analysis

Deep-dive analysis of churn patterns across all dimensions: contract type, tenure, payment method, demographics, and services.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore') if 'warnings' in dir() else None

df = pd.read_csv('../data/processed/cleaned_churn_data.csv')
print(f"Dataset: {df.shape[0]:,} rows · {df.shape[1]} columns")
print(f"Overall churn rate: {df['Churn'].mean()*100:.2f}%")

## 2.1 Churn Overview

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Donut chart
sizes = df['Churn'].value_counts()
axes[0].pie(sizes, labels=['Retained','Churned'], autopct='%1.1f%%',
            colors=['#2dd4bf','#fb7185'], wedgeprops=dict(width=0.5), startangle=90)
axes[0].set_title('Overall Churn Distribution', fontweight='bold')

# Bar by contract
rates = df.groupby('Contract')['Churn'].mean() * 100
axes[1].bar(rates.index, rates.values, color=['#fb7185','#f59e0b','#86efac'])
axes[1].set_title('Churn Rate by Contract Type', fontweight='bold')
axes[1].set_ylabel('Churn Rate (%)')
for i, v in enumerate(rates.values):
    axes[1].text(i, v+0.5, f'{v:.1f}%', ha='center', fontweight='bold')
plt.tight_layout(); plt.show()

## 2.2 Churn by Tenure Cohort

In [ ]:
cohort_rates = df.groupby('TenureCohort', observed=True)['Churn'].mean() * 100
print(cohort_rates)
cohort_rates.plot(kind='bar', color=['#fb7185','#f59e0b','#2dd4bf','#86efac'],
                  figsize=(8,4), rot=0, title='Churn Rate by Tenure Cohort')
plt.ylabel('Churn Rate (%)'); plt.tight_layout(); plt.show()

## 2.3 Payment Method Analysis

In [ ]:
payment = df.groupby('PaymentMethod')['Churn'].mean() * 100
payment.sort_values().plot(kind='barh', figsize=(9,4), color='#fb7185',
                           title='Churn Rate by Payment Method')
plt.xlabel('Churn Rate (%)'); plt.tight_layout(); plt.show()
print(payment.sort_values(ascending=False))

## 2.4 Internet Service Analysis

In [ ]:
inet = df.groupby('InternetService')['Churn'].mean() * 100
print(inet.sort_values(ascending=False))
inet.sort_values().plot(kind='barh', figsize=(7,4),
                        color=['#86efac','#f59e0b','#fb7185'],
                        title='Churn Rate by Internet Service')
plt.xlabel('Churn Rate (%)'); plt.tight_layout(); plt.show()

## 2.5 Monthly Charges Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df[df['Churn']==0]['MonthlyCharges'], bins=30, alpha=0.6,
        color='#2dd4bf', label=f"Retained (mean=${df[df['Churn']==0]['MonthlyCharges'].mean():.2f})")
ax.hist(df[df['Churn']==1]['MonthlyCharges'], bins=30, alpha=0.6,
        color='#fb7185', label=f"Churned (mean=${df[df['Churn']==1]['MonthlyCharges'].mean():.2f})")
ax.legend(); ax.set_xlabel('Monthly Charges ($)'); ax.set_title('Monthly Charges Distribution')
plt.tight_layout(); plt.show()

## 2.6 Correlation Heatmap

In [ ]:
numerics = df.select_dtypes(include='number')
plt.figure(figsize=(12, 8))
mask = np.triu(np.ones_like(numerics.corr(), dtype=bool))
sns.heatmap(numerics.corr(), mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, vmin=-1, vmax=1)
plt.title('Feature Correlation Heatmap', fontweight='bold')
plt.tight_layout(); plt.show()

## 2.7 Demographic Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ['gender', 'SeniorCitizen', 'Partner']):
    rates = df.groupby(col)['Churn'].mean() * 100
    ax.bar(rates.index.astype(str), rates.values, color=['#2dd4bf','#fb7185'])
    ax.set_title(f'Churn by {col}', fontweight='bold')
    ax.set_ylabel('Churn Rate (%)')
plt.tight_layout(); plt.show()

## 2.8 Add-On Services Impact

In [ ]:
services = ['OnlineSecurity','TechSupport','OnlineBackup','DeviceProtection']
results = {}
for svc in services:
    sub = df[df[svc].isin(['Yes','No'])]
    results[svc] = sub.groupby(svc)['Churn'].mean() * 100

results_df = pd.DataFrame(results).T
print(results_df)
results_df.plot(kind='bar', figsize=(10, 5), color=['#86efac','#fb7185'],
                title='Churn Rate — With vs Without Add-On Services')
plt.ylabel('Churn Rate (%)'); plt.xticks(rotation=30); plt.legend(['With','Without'])
plt.tight_layout(); plt.show()

## 2.9 Business Insights Summary

In [ ]:
print("=" * 60)
print("KEY BUSINESS INSIGHTS — TELECOM CHURN ANALYSIS")
print("=" * 60)
insights = [
    ("Overall Churn Rate",               f"{df['Churn'].mean()*100:.2f}%"),
    ("Month-to-Month Contract Churn",    f"{df[df['Contract']=='Month-to-month']['Churn'].mean()*100:.2f}%"),
    ("Two-Year Contract Churn",          f"{df[df['Contract']=='Two year']['Churn'].mean()*100:.2f}%"),
    ("Electronic Check Churn",           f"{df[df['PaymentMethod']=='Electronic check']['Churn'].mean()*100:.2f}%"),
    ("Fiber Optic Churn",                f"{df[df['InternetService']=='Fiber optic']['Churn'].mean()*100:.2f}%"),
    ("New Customer Churn (0-12 mo)",     f"{df[df['TenureCohort']=='0-12 mo']['Churn'].mean()*100:.2f}%" if 'TenureCohort' in df else 'N/A"),
    ("Senior Citizen Churn",             f"{df[df['SeniorCitizen']==1]['Churn'].mean()*100:.2f}%"),
    ("Avg Monthly Charge (Churned)",     f"${df[df['Churn']==1]['MonthlyCharges'].mean():.2f}"),
    ("Avg Monthly Charge (Retained)",    f"${df[df['Churn']==0]['MonthlyCharges'].mean():.2f}"),
]
for label, val in insights:
    print(f"  {label:<35} {val}")